In [1]:
import ast
import os
import pandas as pd

def extract_genus_name(x):
    try:
        genus_dict = ast.literal_eval(x)
        return genus_dict.get('name', None)
    except:
        return None

os.chdir('/active-data/analysis_results/chr_pla')
all_data = pd.read_csv('genome_chr-pla_statistics.csv')
all_data['genus_clean'] = all_data['genus'].apply(extract_genus_name)
counts = all_data['genus_clean'].value_counts()
keep_genus = counts[counts >= 400].index.to_list()
print(keep_genus)

base_folder = f'/active-data/analysis_results/chr_pla/genus'
figure_data = f'{base_folder}/figure_data'
os.makedirs(figure_data, exist_ok=True)

['Escherichia', 'Klebsiella', 'Staphylococcus', 'Pseudomonas', 'Bacillus', 'Salmonella', 'Streptococcus', 'Streptomyces', 'Acinetobacter', 'Enterococcus', 'Bordetella', 'Enterobacter', 'Xanthomonas', 'Campylobacter', 'Vibrio', 'Mycobacterium', 'Corynebacterium', 'Burkholderia', 'Listeria', 'Citrobacter', 'Helicobacter']


In [2]:
from tqdm import tqdm
import warnings
import random
import pandas as pd
import numpy as np
import os
import ast

warnings.filterwarnings("ignore")

# typical_gene_content
def typical_gene_content(genus_name):
    folder = f'{base_folder}/statistics_records/{genus_name}'
    fraction_data = pd.read_csv(f'{folder}/replicon-plasmid_fraction-self_bitscore_statistics.csv')
    
    transit = {'all':0}
    typical = {'all':0}
    chromos = {'all':0}
    head = 'query	seed_ortholog	evalue	score	eggNOG_OGs	max_annot_lvl	COG_category	Description	Preferred_name	GOs	EC	KEGG_ko	KEGG_Pathway	KEGG_Module	KEGG_Reaction	KEGG_rclass	BRITE	KEGG_TC	CAZy	BiGG_Reaction	PFAMs'.split('\t')
    char_gene_pre = ['rep', 'tra', 'par', 'sop', 'dna' ]
    pattern = r'^(' + '|'.join(char_gene_pre) + ')'
    with tqdm(total = len(fraction_data), desc=f'{genus_name}', leave=True, ncols=100, unit='B', unit_scale=True) as pbar:
        for index, row in fraction_data.iterrows():
            contig = row['accession']
            acc = contig.split('-')[0]
            eggnog_file = pd.read_csv(f'/active-data/genomes/bacteria_complete_annotationRefSeq-20250807/eggnog_results/{acc}/{acc}.emapper.annotations', comment = '#', sep = '\t', header = None, names = head)
            eggnog_file = eggnog_file[eggnog_file['query'].str.contains(contig)]
            plasmid_genes = eggnog_file[
                eggnog_file['Preferred_name'].str.contains(pattern, case=False, na=False, regex=True)
            ]
            gene_set = set(plasmid_genes['Preferred_name'])
            if row['category-pident_90'] == 'typical plasmid':
                typical['all'] += 1
                for gene in gene_set:
                    try:
                        typical[gene[:4]] += 1
                    except:
                        typical[gene[:4]] = 1
            elif row['category-pident_90'] == 'intermediate replicon':
                transit['all'] += 1
                for gene in gene_set:
                    try:
                        transit[gene[:4]] += 1
                    except:
                        transit[gene[:4]] = 1
            else:
                chromos['all'] += 1
                for gene in gene_set:
                    try:
                        chromos[gene[:4]] += 1
                    except:
                        chromos[gene[:4]] = 1
                
            pbar.update(1)
    
    os.makedirs(f'{figure_data}/gene_content/{genus_name}', exist_ok=True)
    for name, dict_data in zip(['typical plasmid', 'intermediate replicon', 'typical chromosome'], [typical, transit, chromos]):
        with open(f'{figure_data}/gene_content/{genus_name}/{name}.txt', 'w+') as file:
            file.write(str(dict_data))

def parse_cog_set(s):
    try:
        return ast.literal_eval(s)
    except:
        return set()

def cog_count(genus_name):
    folder = f'{base_folder}/statistics_records/{genus_name}'
    out_folder = os.path.join(figure_data, "cog_count_output", genus_name)
    os.makedirs(out_folder, exist_ok=True)
    
    cog_pd = pd.read_csv(f'{folder}/replicon_cog_set.tsv', sep='\t')
    category_count_df = cog_pd['category-pident_90'].value_counts()
    cog_pd['cog_set_parsed'] = cog_pd['cog_set'].apply(parse_cog_set)
    cog_pd_exploded = cog_pd.explode('cog_set_parsed').dropna(subset=['cog_set_parsed'])
    cog_count = cog_pd_exploded.groupby(['category-pident_90', 'cog_set_parsed']).size().reset_index(name='count')
    result = cog_count.pivot(index='cog_set_parsed', columns='category-pident_90', values='count').fillna(0).astype(int)
    result = result.reset_index()
    
    out_path = os.path.join(out_folder, f"cog_count.csv")
    result.to_csv(out_path, index=False)

In [3]:
# circle_layer_network

def initial_angle_table(acc_set, layer):
    return pd.DataFrame({'acc': list(acc_set), 'angle': np.linspace(-np.pi, np.pi, len(acc_set), endpoint=False), 'layer': [layer]*len(acc_set)})

def inter_layer_table(up_layer, down_layer, link_table):
    temp_link1 = link_table[(link_table['source'].isin(up_layer)) & (link_table['target'].isin(down_layer))]
    temp_link2 = link_table[(link_table['source'].isin(down_layer)) & (link_table['target'].isin(up_layer))]
    return pd.concat([temp_link1, temp_link2], ignore_index=True)

def inner_link_table(layer_set, link_table):
    return link_table[(link_table['source'].isin(layer_set)) & (link_table['target'].isin(layer_set))]

def angle_caculator(item, link_table, angle_table):
    item_pd = link_table[(link_table['source'] == item) | (link_table['target'] == item)]
    link_acc = set(item_pd['source']) | set(item_pd['target'])
    temp_angle_pd = angle_table[angle_table['acc'].isin(link_acc)]
    temp_x = sum(temp_angle_pd['x'])
    temp_y = sum(temp_angle_pd['y'])
    temp_angle = np.arctan2(temp_y, temp_x)
    return temp_angle

def reconstruct_angle(target_pd, refer_pd, inter_layer_link, inner_link=None):
    if inner_link is None or inner_link.empty:
        refer_link = inter_layer_link
        refer_pd = refer_pd.copy()
    else:
        refer_link = pd.concat([inter_layer_link, inner_link])
        refer_pd = refer_pd.copy()
        refer_pd['layer'] = 1000
        refer_pd = pd.concat([target_pd, refer_pd]).copy()
    refer_pd['x'] = refer_pd['layer']*np.cos(refer_pd['angle'])
    refer_pd['y'] = refer_pd['layer']*np.sin(refer_pd['angle'])
    for index in target_pd.index:
        item = target_pd['acc'][index]
        temp_angle = angle_caculator(item, refer_link, refer_pd)
        target_pd.loc[index, 'angle'] = temp_angle
    target_pd = target_pd.sort_values(by='angle', ascending=True)
    
    deg_in_rad = np.pi / 18
    perturbation = np.random.uniform(-deg_in_rad, deg_in_rad, len(target_pd))
    if len(target_pd) > 1:
        target_pd['angle'] = target_pd['angle'] + perturbation
    else:
        target_pd['angle'] = [np.mean(target_pd['angle'])]
    return target_pd

def next_layer_arrangement(down_layer, up_layer, last_angle_table, link_table, layer):
    inter_layer_link = inter_layer_table(up_layer, down_layer, link_table)
    inner_link = inner_link_table(down_layer, link_table)
    angles_table = initial_angle_table(down_layer, layer)
    with tqdm(desc=f'arrangement-{layer}') as pbar:
        while True:
            pre_index = list(angles_table.index)
            angles_table = reconstruct_angle(angles_table, last_angle_table, inter_layer_link) #, inner_link)
            pbar.update(1)
            if list(angles_table.index) == pre_index or pbar.n >= 3:
                break
    if len(inter_layer_link) > 50000:
        source_count = inter_layer_link['source'].value_counts()
        target_count = inter_layer_link['target'].value_counts()
        preserve_source = source_count[source_count <= 5].index
        preserve_target = target_count[target_count <= 5].index
        preserve_link = inter_layer_link[(inter_layer_link['source'].isin(preserve_source)) | (inter_layer_link['target'].isin(preserve_target))]
        sample_link = inter_layer_link[~((inter_layer_link['source'].isin(preserve_source)) | (inter_layer_link['target'].isin(preserve_target)))]
        sample_link = sample_link.sample(n=50000, replace=False)
        inter_layer_link = pd.concat([preserve_link, sample_link])
    return angles_table, inter_layer_link

def process_figure_tables(link_rep_dict, NMS_link, zero_rep, init_rep):
    angle_tables = {}
    link_tables = pd.DataFrame()
    layer_ratio = 1
    
    first_inner_link = inner_link_table(link_rep_dict[1], NMS_link)
    initial_inner_link = inner_link_table(link_rep_dict[0], NMS_link)
    initial_pd = initial_angle_table(link_rep_dict[0], layer=1*layer_ratio)
    
    temp_link = inter_layer_table(zero_rep, link_rep_dict[1], NMS_link)
    
    with tqdm(desc='initiation') as pbar:
        while True:
            pre_index = list(initial_pd.index)
            first_pd = initial_angle_table(link_rep_dict[1], layer=2*layer_ratio)
            first_pd = reconstruct_angle(first_pd, initial_pd, temp_link) #, first_inner_link)
            initial_pd = reconstruct_angle(initial_pd, first_pd, temp_link) #, initial_inner_link)
            pbar.update(1)
            if list(initial_pd.index) == pre_index or pbar.n >= 10:
                break
    angle_tables[0] = initial_pd
    
    initial_rest = link_rep_dict[-1]
    in_re_pd = initial_angle_table(initial_rest, layer=layer_ratio*0.95)
    initial_link = inner_link_table(init_rep, NMS_link)
    link_tables = pd.concat([link_tables, initial_link])
    with tqdm(desc='initial_layer') as pbar:
        while True:
            pre_index = list(in_re_pd.index)
            in_all_pd = pd.concat([initial_pd, in_re_pd])
            in_re_pd = reconstruct_angle(in_re_pd, in_all_pd, initial_link)
            pbar.update(1)
            if list(in_re_pd.index) == pre_index or pbar.n >= 30:
                break
    angle_tables[-1] = in_re_pd
    
    for i in range(1, max(link_rep_dict.keys())):
        angle_tables[i], inter_layer_link = next_layer_arrangement(link_rep_dict[i], link_rep_dict[i-1], angle_tables[i-1], NMS_link, layer=(i+1)*layer_ratio)
        link_tables = pd.concat([link_tables, inter_layer_link])
    
    filted_link_table = link_tables
    
    all_angle_table = pd.concat([angle_tables[key] for key in angle_tables.keys()], ignore_index=True)
    all_angle_table['layer'] = all_angle_table['layer'] - layer_ratio * 0.3
    radius = np.random.uniform(-layer_ratio*0.1, layer_ratio*0.1, len(all_angle_table))
    all_angle_table['layer_f'] = all_angle_table['layer'] + radius
    all_angle_table['x'] = all_angle_table['layer_f']*np.cos(all_angle_table['angle'])
    all_angle_table['y'] = all_angle_table['layer_f']*np.sin(all_angle_table['angle'])
    
    merge_source = pd.merge(left=filted_link_table, right=all_angle_table[['acc', 'x', 'y']], left_on='source', right_on='acc', how='left')
    merged = merge_source[['source', 'target', 'coverage', 'x', 'y']]
    merged.columns = ['source', 'target', 'coverage', 'source_x', 'source_y']
    merge_target = pd.merge(left=merged, right=all_angle_table[['acc', 'x', 'y']], left_on='target', right_on='acc', how='left')
    merged = merge_target[['source', 'target', 'coverage', 'source_x', 'source_y', 'x', 'y']]
    merged.columns = ['source', 'target', 'coverage', 'source_x', 'source_y', 'target_x', 'target_y']
    
    return all_angle_table, merged

def circle_layer_network(genus_name):
    folder = f'{base_folder}/statistics_records/{genus_name}'
    label = 'pident_90'
    
    plas_frac = pd.read_csv(f'{folder}/replicon-plasmid_fraction-self_bitscore_statistics.csv')
    plas_frac = plas_frac[plas_frac[f'category-{label}'].isin(['typical plasmid', 'intermediate replicon'])]
    
    NMS_link = pd.read_csv(f'{folder}/{genus_name}_NMS_replicon_link.csv')
    # NMS_link = NMS_link[NMS_link['coverage'] > 0.9]
    NMS_link = NMS_link[(NMS_link['source'].isin(plas_frac['accession'])) & (NMS_link['target'].isin(plas_frac['accession']))]
    
    link_replicon = set(NMS_link['source']) | set(NMS_link['target'])
    print(genus_name, 'link_replicon:', len(link_replicon))
    plas_frac = plas_frac[plas_frac['accession'].isin(link_replicon)][['organism', 'accession', 'size', f'average plasmid fraction-{label}', 'cp-label', f'category-{label}']]
    plas_frac['log10_size'] = np.log10(plas_frac['size'])
    
    init_rep = set(plas_frac[plas_frac[f'average plasmid fraction-{label}'] < 0.3]['accession'])
    
    link_rep_dict = {0: init_rep}
    i = 0
    temp_set = init_rep
    
    while True:
        next_link = NMS_link[(NMS_link['source'].isin(link_rep_dict[i])) | (NMS_link['target'].isin(link_rep_dict[i]))]
        next_rep = (set(next_link['source']) | set(next_link['target'])) - temp_set
        if len(next_rep) == 0:
            break
        link_rep_dict[i+1] = next_rep
        temp_set = temp_set | next_rep
        i += 1
        
    zero_link = NMS_link[(NMS_link['source'].isin(link_rep_dict[1])) | (NMS_link['target'].isin(link_rep_dict[1]))]
    zero_rep = (set(zero_link['source']) | set(zero_link['target'])) & init_rep
    link_rep_dict[0] = zero_rep
    link_rep_dict[-1] = init_rep - zero_rep
    
    all_angle_table, merged_pd = process_figure_tables(link_rep_dict, NMS_link, zero_rep, init_rep)
    
    out_folder = os.path.join(figure_data, "circle_layer_network_output", genus_name)
    os.makedirs(out_folder, exist_ok=True)

    all_angle_table_path = os.path.join(out_folder, "all_dot_table.csv")
    all_angle_table.to_csv(all_angle_table_path, index=False)

    merged_pd_path = os.path.join(out_folder, "all_link_table.csv")
    merged_pd.to_csv(merged_pd_path, index=False)

In [4]:
for genus_name in keep_genus:
    typical_gene_content(genus_name)
    %time cog_count(genus_name)

Escherichia: 100%|██████████████████████████████████████████████| 15.4k/15.4k [10:13<00:00, 25.1B/s]


CPU times: user 29.3 s, sys: 5.55 s, total: 34.9 s
Wall time: 35 s


Klebsiella: 100%|███████████████████████████████████████████████| 15.0k/15.0k [09:36<00:00, 26.0B/s]


CPU times: user 23 s, sys: 3.56 s, total: 26.6 s
Wall time: 26.7 s


Staphylococcus: 100%|███████████████████████████████████████████| 5.08k/5.08k [01:32<00:00, 55.1B/s]


...

CPU times: user 1.97 s, sys: 2.09 ms, total: 1.97 s
Wall time: 1.99 s


Citrobacter: 100%|██████████████████████████████████████████████| 1.48k/1.48k [00:54<00:00, 27.3B/s]


CPU times: user 2.38 s, sys: 10.7 ms, total: 2.39 s
Wall time: 2.41 s


Helicobacter: 100%|█████████████████████████████████████████████████| 539/539 [00:07<00:00, 76.3B/s]


CPU times: user 870 ms, sys: 5.07 ms, total: 875 ms
Wall time: 887 ms


In [5]:
for genus_name in keep_genus:
    try:
        circle_layer_network(genus_name)
    except:
        pass

Escherichia link_replicon: 11059


initiation: 10it [00:36,  3.61s/it]
initial_layer: 30it [00:01, 16.65it/s]
arrangement-2: 3it [00:10,  3.57s/it]
arrangement-3: 3it [13:55, 278.66s/it]
arrangement-4: 3it [00:12,  4.15s/it]
arrangement-5: 3it [00:03,  1.24s/it]
arrangement-6: 3it [00:01,  1.71it/s]
arrangement-7: 3it [00:01,  2.03it/s]
arrangement-8: 2it [00:00, 15.00it/s]


Klebsiella link_replicon: 11217


initiation: 10it [00:41,  4.19s/it]
initial_layer: 30it [00:01, 17.77it/s]
arrangement-2: 3it [00:12,  4.14s/it]
arrangement-3: 3it [24:49, 496.35s/it]
arrangement-4: 3it [00:08,  2.85s/it]
arrangement-5: 2it [00:00,  5.92it/s]
arrangement-6: 2it [00:00, 24.52it/s]


Staphylococcus link_replicon: 2367


...

Bordetella link_replicon: 0
Enterobacter link_replicon: 1921


initiation: 1it [00:00, 20.49it/s]
initial_layer: 2it [00:00, 308.54it/s]
arrangement-2: 2it [00:00, 22.28it/s]
arrangement-3: 3it [00:04,  1.37s/it]
arrangement-4: 3it [00:00,  3.46it/s]
arrangement-5: 3it [00:00,  4.87it/s]
arrangement-6: 2it [00:00, 50.97it/s]


Xanthomonas link_replicon: 530
Campylobacter link_replicon: 338


initiation: 10it [00:00, 36.57it/s]
initial_layer: 2it [00:00, 315.50it/s]
arrangement-2: 2it [00:00, 46.92it/s]
arrangement-3: 2it [00:00, 10.64it/s]
arrangement-4: 2it [00:00, 266.54it/s]
arrangement-5: 1it [00:00, 78.88it/s]
arrangement-6: 2it [00:00, 66.08it/s]


Vibrio link_replicon: 352
Mycobacterium link_replicon: 118
Corynebacterium link_replicon: 32
Burkholderia link_replicon: 274
Listeria link_replicon: 130
Citrobacter link_replicon: 955
Helicobacter link_replicon: 81
